# Supervised Learning and the Linear Model

> 📖 Book: [ch01](../../.claude/skills/ml-knowledge/chapters/ch01-introduction.md), [ch04](../../.claude/skills/ml-knowledge/chapters/ch04-linear-regression-gradient-descent.md) · Prev: [01](01_what_is_ml.ipynb) · Next: [03 Linear inversion](03_linear_inversion_problem.ipynb)

Supervised learning uses input–output pairs to find a model that maps each input to the correct output as well as possible. A well-trained model predicts outputs for new, previously unseen inputs.

Given a training set, the goal is to learn a function $h: \mathcal{X} \to \mathcal{Y}$ that gives a "good" prediction of $y$. When $y$ is continuous we have a **regression problem**; when $y$ takes a small number of discrete values we have a **classification problem**.

## Notation

A **model** is a simplified conceptual construction of a more complex reality, described with mathematical language.

- $\mathbf{x}^{(i)} \in \mathbb{R}^n$ – input (feature) vector of the $i$-th sample, e.g. the size of a house.
- $y^{(i)}$ – output / target / label, e.g. the price of the house.
- $(\mathbf{x}^{(i)}, y^{(i)})$ – a training pair, $i = 1, \dots, m$; $\mathcal{D} = \{(\mathbf{x}^{(i)}, y^{(i)})\}_{i=1}^m$ is the **training set**.
- $m$ – number of training samples; $n$ – number of features.
- $X \in \mathbb{R}^{m \times n}$ – all inputs stacked by rows; $\mathcal{X}$, $\mathcal{Y}$ – input and output spaces.
- $h_\theta$ – the **hypothesis**: the model's conjecture of how $y$ depends on $\mathbf{x}$, parametrized by $\theta$.

## Linear model with a single variable

$$
h_\theta(x) = \theta_0 + \theta_1 x
$$

or in vector form, prepending a $1$ for the intercept (bias):

$$
h_\theta(\mathbf{x}) = \mathbf{x}^T\theta =
\begin{pmatrix} 1 & x \end{pmatrix}
\begin{pmatrix} \theta_0 \\ \theta_1 \end{pmatrix}.
$$

With $n$ features, $h_\theta(\mathbf{x}) = \theta_0 + \theta_1 x_1 + \dots + \theta_n x_n = \mathbf{x}^T\theta$. By choosing $\theta$ appropriately, $h_\theta$ can model the trend of the data.

In [1]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(0)

## Try it: fit a line by hand

Move the sliders to pick $\theta_0$ (intercept) and $\theta_1$ (slope). The right panel shows the **cost** $J(\theta) = \frac{1}{2m}\sum_i (h_\theta(x^{(i)}) - y^{(i)})^2$: the average squared vertical distance (red segments) between the line and the points. Learning = finding the $\theta$ that makes $J$ smallest.

In [2]:
x = 10 * rng.random(50)
y = 2 * x - 5 + rng.normal(size=50)


def cost(t0, t1):
    return np.mean((t0 + t1 * x - y) ** 2) / 2


@widgets.interact(theta0=(-10.0, 10.0), theta1=(-3.0, 3.0))
def plot(theta0=0.0, theta1=1.0):
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    h = theta0 + theta1 * x
    ax.vlines(x, y, h, color="r", alpha=0.3)
    ax.scatter(x, y, label="data")
    xs = np.linspace(0, 10, 2)
    ax.plot(xs, theta0 + theta1 * xs, "k", label=rf"$h_\theta(x) = {theta0:.1f} + {theta1:.2f}x$")
    ax.set(xlabel="$x$", ylabel="$y$", title=rf"$J(\theta) = {cost(theta0, theta1):.2f}$")
    ax.legend()
    T0, T1 = np.meshgrid(np.linspace(-10, 10, 100), np.linspace(-3, 3, 100))
    J = np.vectorize(cost)(T0, T1)
    ax2.contour(T0, T1, J, levels=np.logspace(-0.5, 3, 25), cmap="viridis")
    ax2.plot(theta0, theta1, "r*", ms=15)
    ax2.set(xlabel=r"$\theta_0$", ylabel=r"$\theta_1$", title=r"$J(\theta_0, \theta_1)$")
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='theta0', max=10.0, min=-10.0), FloatSlider(value=1.0…

## The least-squares problem

Stacking the samples, we want $\theta$ such that $X\theta = y$ with

$$
X =
\begin{pmatrix}
1 & x_1^{(1)} & \cdots & x_n^{(1)} \\
\vdots & \vdots & \ddots & \vdots \\
1 & x_1^{(m)} & \cdots & x_n^{(m)}
\end{pmatrix}, \quad
\theta = \begin{pmatrix} \theta_0 \\ \vdots \\ \theta_n \end{pmatrix}, \quad
y = \begin{pmatrix} y^{(1)} \\ \vdots \\ y^{(m)} \end{pmatrix}.
$$

With more samples than parameters ($m > n+1$) and noisy data, $y$ is generally **not** in the range of $X$, so no exact solution exists. We settle for the $\theta$ minimizing the squared error

$$
\min_\theta \|X\theta - y\|^2 = \min_\theta \sum_{i=1}^m \left(\mathbf{x}^{(i)T}\theta - y^{(i)}\right)^2 ,
$$

which leads to the **normal equations** $X^TX\theta = X^Ty$ (next: [01_linear_regression/01_normal_equations](../01_linear_regression/01_normal_equations.ipynb)).

Linear regression assumes linearity, independent errors and homoscedasticity; check them before trusting the model.